# Suite VIS — Visitors protocol conformance

Every class that plays a visitor role must implement its `Visitors` protocol completely, with matching arity:
Proxies builders, Plain writers, the Reachable collector, the Validators recorder, and the Comparison recorders. Proxies instances implement
`Visitable`, not a visitor protocol. TypeScript checks `implements` at compile time; these cases check at runtime.

In [ ]:
import { Comparison, Plain, Proxies, Reachable, Schemas as S, Validators, Visitors } from "@mbse/schemas/Framework";
import { NotImplementedError } from "@mbse/schemas/Framework/Errors";
import { assert, conformance_problems, instance_problems, PROTOCOLS, raises, same, text } from "./support.js";

## VIS-01 · The protocols declare what FRAMEWORK.md says

In [ ]:
assert(same(Object.keys(Visitors.PROTOCOL_METHODS).sort(), Object.keys(PROTOCOLS).sort()));
for (const [name, methods] of Object.entries(Visitors.PROTOCOL_METHODS)) {
  assert(same([...methods].sort(), Object.keys(PROTOCOLS[name] as object).sort()), name);
}
assert(same([...Visitors.PROTOCOL_METHODS.OfAny], ["as_native", "as_object", "as_union", "as_intersection"]));
assert(same([...Visitors.PROTOCOL_METHODS.Visitable], ["identity", "schema_name", "accept"]));
assert(!(Visitors.PROTOCOL_METHODS.OfObject as readonly string[]).includes("identity")); // identity belongs to the visited object
{
  // Native is exactly bigint | number | string | boolean | Uint8Array (a compile-time check).
  const natives: Visitors.Native[] = [1n, 1.5, "s", true, new Uint8Array()];
  assert(natives.length === 5);
}

## VIS-02 · Every implementation conforms to its protocol

In [ ]:
const P = Proxies._internals;
const implementations: Record<string, [{ name: string; prototype: object }, string][]> = {
  Proxies: [[P.ObjectBuilderTarget, "OfObject"], [P._AdjacencySlot, "OfAdjacency"], [P._EntryBuilder, "OfEntry"],
    [P._LinkSlot, "OfLink"], [P._PropertySlot, "OfProperty"], [P._AnySlot, "OfAny"], [P._NativeSlot, "OfNative"],
    [P.ObjectTarget, "Visitable"], [P.RecordBuilderTarget, "OfObject"], [P._UnionSlot, "OfUnion"],
    [P._IntersectionSlot, "OfIntersection"]],
  Plain: [[Plain._ObjectWriter, "OfObject"], [Plain._AdjacencyWriter, "OfAdjacency"], [Plain._EntryWriter, "OfEntry"],
    [Plain._LinkWriter, "OfLink"], [Plain._PropertyWriter, "OfProperty"], [Plain._AnyWriter, "OfAny"], [Plain._NativeWriter, "OfNative"],
    [Plain._UnionWriter, "OfUnion"], [Plain._IntersectionWriter, "OfIntersection"]],
  Reachable: [[Reachable._Collector, "OfObject"], [Reachable._Adjacency, "OfAdjacency"], [Reachable._Entry, "OfEntry"],
    [Reachable._Link, "OfLink"], [Reachable._Ignored, "OfProperty"], [Reachable._Ignored, "OfAny"], [Reachable._Ignored, "OfNative"]],
  Validators: [[Validators._ObjectRecord, "OfObject"], [Validators._AdjacencyRecord, "OfAdjacency"],
    [Validators._EntryRecord, "OfEntry"], [Validators._Link, "OfLink"], [Validators._Value, "OfProperty"],
    [Validators._Value, "OfAny"], [Validators._Value, "OfNative"], [Validators._UnionRecord, "OfUnion"],
    [Validators._IntersectionRecord, "OfIntersection"]],
  Comparison: [[Comparison.OfObject, "OfObject"], [Comparison.OfAdjacency, "OfAdjacency"], [Comparison.OfEntry, "OfEntry"],
    [Comparison.OfLink, "OfLink"], [Comparison.OfProperty, "OfProperty"], [Comparison.OfAny, "OfAny"],
    [Comparison.OfNative, "OfNative"], [Comparison.OfUnion, "OfUnion"], [Comparison.OfIntersection, "OfIntersection"]],
};
{
  const problems = Object.values(implementations).flat().flatMap(([impl, proto]) => conformance_problems(impl, proto));
  assert(problems.length === 0, problems.join("; "));
}

## VIS-03 · Proxies instances are `Visitable`, not visitors
FRAMEWORK.md: builders and serializers implement `Visitors`; proxies do not.

In [ ]:
for (const protocol of ["OfObject", "OfAdjacency", "OfEntry"]) {
  assert(conformance_problems(P.ObjectTarget, protocol).length > 0, protocol);
}

## VIS-04 · Visitor methods chain: non-query methods return `this`

In [ ]:
const Rel = new S.OfRelation.Builder().links("a", "b").properties(text("k")).create();
const Obj = new S.OfObject.Builder().properties(text("x")).relations((r) => r.name("r").of(Rel).me("a")).create();
Proxies.register("Obj", Obj);
Proxies.register("Rel", Rel);
{
  const builder = Proxies.Builders.Obj();
  assert(builder.property("x", () => null) === builder);
  assert(builder.properties(() => null) === builder);
  assert(builder.clear("x") === builder);
  assert(builder.adjacencies(() => null) === builder);
  assert(builder.adjacency("r", () => null) === builder);

  const slots: any[] = [];
  builder.property("x", (p: any) => slots.push(p));
  const prop = slots[0];
  assert(prop.value(() => null) === prop && prop.clear() === prop);
  prop.value((a: any) => slots.push(a));
  const any_ = slots[1];
  assert(any_.as_native(() => null) === any_);
  any_.as_native((n: any) => slots.push(n));
  const native = slots[2];
  assert(native.set("v") === native && native.clear() === native);

  builder.adjacency("r", (a: any) => slots.push(a));
  const adjacency = slots[3];
  assert(adjacency.add((e: any) => slots.push(e)) === adjacency);
  const entry = slots[4];
  assert(entry.property("k", () => null) === entry && entry.clear("k") === entry);
  assert(entry.links((link: any) => slots.push(link)) === entry);
  const link = slots[5];
  assert(link.name() === "b" && link.set(Proxies.Builders.Obj().create()) === link);
  assert(adjacency.remove(entry) === adjacency && adjacency.entries(() => null) === adjacency);
}

## VIS-05 · `OfRelation` has no implementation yet; intersections of unions are refused
`OfRelation` is declared but no class implements it. Intersections of unions have no merged schema yet, so every
implementation refuses their values (pinned here, see TestPlan.md).

In [ ]:
const Id = new S.OfUnion.Builder().branches((b) => b.of((t) => t.as_native(BigInt)).when("i"),
  (b) => b.of((t) => t.as_native(String)).when("s")).create();
const Ids = new S.OfIntersection.Builder().of(Id, Id).create();
assert(Ids.validate().length === 0);
raises(NotImplementedError, () => Ids.merged(), "intersections of unions are not supported yet");

## VIS-06 · Live instances conform too
A class can declare every method yet have an instance field shadow one of them (the Python recorder once did:
its `adjacencies` map hid its `adjacencies()` method). This checks method callability on real instances.

In [ ]:
{
  const found: [object, string][] = [];
  const builder = Proxies.Builders.Obj();
  found.push([builder, "OfObject"]);
  builder.property("x", (p: any) => { found.push([p, "OfProperty"]); p.value((a: any) => { found.push([a, "OfAny"]); a.as_native((n: any) => found.push([n, "OfNative"])); }); });
  builder.adjacency("r", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => { found.push([e, "OfEntry"]); e.link("b", (l: any) => found.push([l, "OfLink"])); }); });
  found.push([Proxies.Builders.Obj().create(), "Visitable"]);
  const record = new Validators._ObjectRecord();
  record.adjacency("r", (a: any) => a.add((e: any) => found.push([e, "OfEntry"])));
  found.push([record, "OfObject"], [new Reachable._Collector(() => null), "OfObject"]);
  const writer = new Plain._ObjectWriter(new Map(), Obj, () => new Map());
  found.push([writer, "OfObject"]);
  writer.adjacency("r", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => found.push([e, "OfEntry"])); });
  const comparison = new Comparison.OfObject(Obj);
  found.push([comparison, "OfObject"]);
  comparison.property("x", (p: any) => { found.push([p, "OfProperty"]); p.value((a: any) => { found.push([a, "OfAny"]); a.as_native((n: any) => found.push([n, "OfNative"])); }); });
  comparison.adjacency("r", (a: any) => { found.push([a, "OfAdjacency"]); a.add((e: any) => { found.push([e, "OfEntry"]); e.link("b", (l: any) => found.push([l, "OfLink"])); }); });
  const problems = found.flatMap(([instance, protocol]) => instance_problems(instance, protocol));
  assert(problems.length === 0, problems.join("; "));
  assert(record.adjacencies((a) => a.name()) === record); // the method the Python bug hid
}

## VIS-07 · Every protocol method of every implementation behaves
Serializers, the reachability collector and the validation recorder only receive the calls a `Visitable` makes.
This calls every other protocol method on each of them, and on the parts of the proxy builder that the DSL skips.

In [ ]:
import { AttributeError, KeyError } from "@mbse/schemas/Framework/Errors";
import { equal, map } from "./support.js";
{
  const target = Proxies.Builders.Obj().create();
  const seen: unknown[] = [];
  const pairs = (...xs: unknown[]) => xs;

  // Plain writers: every method on objects, properties, values, adjacencies, entries and links.
  const out = new Map();
  const writer = new Plain._ObjectWriter(out, Obj, (t) => map({ $ref: "s9", $schema: t.schema_name() }));
  assert(!writer.has("x"));
  writer.properties((p) => seen.push(p)); // nothing is present yet
  assert(seen.length === 0);
  writer.property("x", (p) => p.value((a) => a.as_native((n) => n.set("v"))));
  assert(writer.has("x") && equal(out, map({ x: "v" })));
  writer.properties((p) => seen.push(pairs(p.name(), p.has())));
  writer.property("x", (p) => p.value((a) => a.as_native((n) => seen.push(pairs(n.has(), n.get())))));
  writer.property("x", (p) => p.value((a) => a.as_native((n) => n.clear())));
  assert(out.size === 0 && equal(seen, [["x", true], [true, "v"]]));
  writer.property("x", (p) => p.value((a) => a.as_native((n) => n.set("w"))).clear());
  writer.property("x", (p) => p.value((a) => a.as_native((n) => n.set("w"))));
  assert(writer.clear("x") === writer && out.size === 0);
  raises(KeyError, () => writer.property("nope", () => null), "unknown property");

  const fill = (entry: any) => {
    entry.properties((p: unknown) => seen.push(p)); // nothing is present yet
    entry.links((k: any) => { seen.push(k.name()); k.set(target); });
    raises(NotImplementedError, () => entry.link("b", (k: any) => k.target(() => null)));
    raises(KeyError, () => entry.link("a", () => null), "not a link this entry can set"); // the object's own link
    entry.property("k", (p: any) => p.value((v: any) => v.as_native((n: any) => n.set("K"))));
    seen.push(entry.has("k"));
    entry.properties((p: any) => seen.push(p.name()));
  };
  seen.length = 0;
  writer.adjacencies((a) => { seen.push(pairs(a.name(), a.me())); a.add(fill); });
  assert(equal(seen, [["r", "a"], "b", true, "k"]));
  assert(equal(out, map({ r: [{ b: { $ref: "s9", $schema: "Obj" }, k: "K" }] })));
  writer.adjacency("r", (a) => a.entries((e) => e.clear("k")));
  assert(equal(out, map({ r: [{ b: { $ref: "s9", $schema: "Obj" } }] })));
  raises(NotImplementedError, () => writer.adjacency("r", (a) => a.remove(null as never)));

  // The reachability collector ignores values and reports only link targets.
  const found: unknown[] = [];
  const collector = new Reachable._Collector((t) => found.push(t));
  assert(!collector.has("x") && collector.clear("x") === collector);
  assert(collector.properties((p) => found.push(p)) === collector && collector.adjacencies((a) => found.push(a)) === collector);
  collector.property("x", (p) => {
    seen.push(p.name(), p.has());
    p.clear();
    p.value((a) => a.as_native((n) => { n.set(1n); n.clear(); }));
  });
  for (const method of ["as_object", "as_union", "as_intersection"] as const) {
    collector.property("x", (p) => p.value((a) => a[method]((v: unknown) => found.push(v))));
  }
  const collect = (entry: any) => {
    assert(entry.links((l: unknown) => found.push(l)) === entry && entry.properties((p: unknown) => found.push(p)) === entry);
    assert(!entry.has("k") && entry.clear("k") === entry);
    entry.property("k", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("K"))));
    entry.link("b", (k: any) => { seen.push(k.name()); k.set(target); });
  };
  collector.adjacency("r", (a) => { seen.push(a.name()); a.entries((e) => found.push(e)); a.add(collect); a.remove(null as never); });
  assert(same(found, [target])); // only the link target was reported

  // The validation recorder keeps everything it is given, and can be read back through the same protocols.
  const record = new Validators._ObjectRecord();
  record.property("x", (p) => p.value((a) => a.as_native((n) => n.set("v"))));
  seen.length = 0;
  record.properties((p) => seen.push(pairs(p.name(), p.has())));
  record.property("x", (p) => p.value((a) => a.as_native((n) => seen.push(n.get()))));
  assert(record.has("x") && equal(seen, [["x", true], "v"]));
  record.property("x", (p) => p.clear());
  assert(!record.has("x"));
  record.property("x", (p) => p.value((a) => a.as_native((n) => n.set("v").clear())));
  assert(record.clear("x") === record && !record.has("x"));
  record.adjacency("r", (a) => a.add((entry) => {
    entry.link("b", (k) => k.set(target));
    entry.property("k", (p) => p.value((a2) => a2.as_native((n) => n.set("K"))));
  }));
  seen.length = 0;
  record.adjacencies((a) => {
    seen.push(a.name());
    a.entries((entry) => {
      entry.links((k) => { seen.push(k.name()); k.target((t) => seen.push(t)); });
      entry.properties((p) => seen.push(p.name()));
      seen.push(entry.has("k"));
      entry.clear("k");
      seen.push(entry.has("k"));
    });
  });
  assert(same(seen, ["r", "b", target, "k", true, false]));
  raises(NotImplementedError, () => record.adjacency("r", (a) => a.me()));
  record.adjacency("r", (a) => a.entries((e) => a.remove(e)));
  record.adjacencies((a) => a.entries(() => seen.push("left")));
  assert(!seen.includes("left"));

  // Proxy builders: the entry and slot methods the DSL does not use.
  seen.length = 0;
  const builder = Proxies.Builders.Obj();
  builder.adjacency("r", (a: any) => { a.entries((e: unknown) => seen.push(e)); a.remove(null); }); // no entries yet
  assert(builder.then === undefined && typeof String(builder) === "string" && builder[Symbol.toStringTag] === undefined);
  builder.adjacency("r", (a: any) => a.add((entry: any) => {
    entry.links((k: any) => { seen.push(k.name()); k.set(target); k.target((t: unknown) => seen.push(t)); });
    entry.property("k", (p: any) => p.value((a2: any) => a2.as_native((n: any) => n.set("K"))));
    entry.properties((p: any) => seen.push(pairs(p.name(), p.has())));
    seen.push(entry.clear("k").has("k"));
    raises(AttributeError, () => entry.property("nope", () => null), "not a property of this relation");
    raises(AttributeError, () => entry._private);
    raises(AttributeError, () => entry.nope);
    assert(entry.then === undefined && entry[Symbol.toStringTag] === undefined && typeof String(entry) === "string");
  }));
  assert(equal(seen, ["b", target, ["k", true], false]));
  raises(AttributeError, () => builder.property("x", (p: any) => p.value((a: any) => a.as_native((n: any) => n.get()))), "'x' is not set");
  builder.property("x", (p: any) => { seen.push(p.has()); p.value((a: any) => a.as_native((n: any) => n.set("v"))); p.clear(); });
  assert(!builder.has("x"));
}